# JSMA: Jacobian and Gradients

This notebook follows the HTB Academy **Jacobian and Gradients** subsection. We build the reusable pieces needed by the Jacobian-based Saliency Map Attack (JSMA): one-class input gradients, the full Jacobian, target/competitor extraction, and search-space masking.

JSMA is targeted: it asks which input pixels can increase a chosen target logit while decreasing the other logits. The attack loop and saliency formula come in later subsections.

## Mathematical map

For class score `F_i(x)` and pixel `x_j`, the Jacobian entry is

`J_ij = partial F_i / partial x_j`

Read this aloud as: **J sub i-j equals the partial derivative of F sub i with respect to x sub j**. It means how much class `i`'s score changes when pixel `j` changes slightly. For `m` classes and `n` input features, `J` has shape `(m, n)`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from htb_ai_library.core import set_reproducibility
from htb_ai_library.data import get_mnist_loaders
from htb_ai_library.models import SimpleLeNet
from htb_ai_library.training import train_model
from htb_ai_library.utils import save_model, load_model
from htb_ai_library.visualization import use_htb_style

use_htb_style()
set_reproducibility(1337)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

The next cell loads the same LeNet-like MNIST model used by HTB. A cached checkpoint avoids retraining on every run. `eval()` disables training-only behavior such as dropout.

In [ ]:
train_loader, test_loader = get_mnist_loaders(batch_size=128)
model_path = output_dir / 'mnist_target.pth'
model = SimpleLeNet().to(device)

if model_path.exists():
    print(f'Loading existing model from {model_path}')
    model = load_model(model, model_path, device)
else:
    print('Training new model...')
    model = train_model(model, train_loader, test_loader, epochs=5, learning_rate=0.001, device=device)
    save_model(model, model_path)

model.eval()
print('Model ready for JSMA gradients')

## One class gradient

Autograd needs one scalar output. Selecting `logits[0, class_idx]` means we differentiate one image's one class score with respect to every input pixel. We use logits rather than softmax probabilities so target and competitor sensitivities remain independent.

In [ ]:
def compute_class_gradient(x, model, class_idx, wrt='logits'):
    """Return d(selected class score) / d(input) as a flat NumPy vector."""
    if x.shape[0] != 1:
        raise ValueError('compute_class_gradient expects batch size 1')

    x_grad = x.detach().clone().requires_grad_(True)
    logits = model(x_grad)

    if wrt == 'logits':
        scalar = logits[0, class_idx]
    elif wrt == 'probabilities':
        probs = F.softmax(logits, dim=1)
        scalar = probs[0, class_idx]
    else:
        raise ValueError("wrt must be 'logits' or 'probabilities'")

    scalar.backward()
    return x_grad.grad.detach().cpu().numpy().flatten().copy()

In [ ]:
# A deterministic four-feature toy model makes the gradient shape visible.
model_test = nn.Sequential(nn.Flatten(), nn.Linear(4, 2)).eval()
x_test = torch.tensor([[[[0.5, 0.3], [0.2, 0.8]]]])
grad_class0 = compute_class_gradient(x_test, model_test, 0)
print('Gradient shape:', grad_class0.shape)
print('Gradient for class 0:', grad_class0)

A positive gradient component says increasing that feature raises the selected score; a negative component says decreasing it may help. Flattening preserves channel-height-width order, so the vector index maps consistently back to the image.

In [ ]:
def compute_jacobian_matrix(x, model, num_classes=10, wrt='logits'):
    """Return a (num_classes, num_features) Jacobian for one image."""
    if x.shape[0] != 1:
        raise ValueError('compute_jacobian_matrix expects batch size 1')

    rows = [
        compute_class_gradient(x, model, class_idx, wrt)
        for class_idx in range(num_classes)
    ]
    return np.asarray(rows)

print('For MNIST:', 10 * 28 * 28, 'Jacobian values')
print('Expected shape:', (10, 784))

Each Jacobian row is one class gradient. The explicit batch-size check prevents accidentally computing a result that does not correspond to one image's per-sample sensitivities.

In [ ]:
def extract_target_gradient(jacobian, target_class):
    return jacobian[target_class].copy()


def extract_other_gradients(jacobian, target_class):
    total_grad = jacobian.sum(axis=0)
    return total_grad - jacobian[target_class]

J_toy = np.array([
    [0.2, 0.5, -0.1, 0.3],
    [-0.1, 0.2, 0.4, -0.2],
    [0.6, -0.3, 0.1, 0.5],
])
target = 2
alpha = extract_target_gradient(J_toy, target)
beta = extract_other_gradients(J_toy, target)
print('Target gradient alpha:', alpha)
print('Other-gradient sum beta:', beta)

Read `alpha` as the target class's sensitivity vector. Read `beta` as the combined sensitivity of every non-target class. In later saliency code, JSMA looks for directions where alpha is positive and beta is negative (or the reverse direction).

In [ ]:
def apply_search_mask(gradient, search_space):
    """Zero unavailable features without changing their indices."""
    return gradient * search_space

grad = np.array([0.5, -0.2, 0.8, 0.1, -0.4])
mask = np.array([True, False, True, False, True])
masked_grad = apply_search_mask(grad, mask)
print('Original gradient:', grad)
print('Search-space mask:', mask)
print('Masked gradient:  ', masked_grad)

Multiplication converts `True` to 1 and `False` to 0. We keep the vector length and therefore preserve pixel-to-index correspondence. This mask can exclude pixels that are already saturated at `clip_min`/`clip_max` or have already been consumed by the attack.

**Notebook boundary:** this subsection stops here. Saliency scoring and feature selection belong to the next HTB subsection.

## Saliency scoring

For feature `j`, `alpha_j` (say **alpha sub j**) is the target gradient and `beta_j` (say **beta sub j**) is the combined competitor gradient. For an increase, `alpha_j > 0` and `beta_j < 0`; the score is `|alpha_j| × |beta_j|` when valid, otherwise zero. We score increase and decrease directions separately because reversing a pixel reverses the useful sign pattern.

In [ ]:
def score_increase_saliency(target_grad, other_grad):
    increase_mask = (target_grad > 0) & (other_grad < 0)
    return target_grad * np.abs(other_grad) * increase_mask

def score_decrease_saliency(target_grad, other_grad):
    decrease_mask = (target_grad < 0) & (other_grad > 0)
    return np.abs(target_grad) * other_grad * decrease_mask

alpha = np.array([0.6, -0.3, 0.4, 0.1, -0.5, 0.2])
beta = np.array([-0.2, 0.4, -0.5, 0.3, 0.6, -0.1])
inc_scores = score_increase_saliency(alpha, beta)
dec_scores = score_decrease_saliency(alpha, beta)
print('Increase scores:', inc_scores)
print('Decrease scores:', dec_scores)

A zero score means a feature failed the sign test, not necessarily that its gradients were small. A valid feature must help the target and suppress competitors in the tested direction.

In [ ]:
def select_best_direction(inc_scores, dec_scores):
    max_inc_idx = int(np.argmax(inc_scores))
    max_dec_idx = int(np.argmax(dec_scores))
    max_inc_score = float(inc_scores[max_inc_idx])
    max_dec_score = float(dec_scores[max_dec_idx])
    if max_inc_score > max_dec_score:
        return max_inc_idx, max_inc_score, True
    return max_dec_idx, max_dec_score, False

pixel_idx, score, increase = select_best_direction(inc_scores, dec_scores)
direction = 'increase' if increase else 'decrease'
print(f'Selected pixel {pixel_idx}, score {score:.3f}, {direction}')

`argmax` is pronounced **argument max**: it returns the index where the largest value occurs. We compare the best increase against the best decrease. If every score is zero, the caller must stop instead of modifying the arbitrary index returned by `argmax`.

In [ ]:
def initialize_search_space(shape):
    num_features = int(np.prod(shape[1:]))
    return np.ones(num_features, dtype=bool)

def remove_saturated_pixels(search_space, x, clip_min=0.0, clip_max=1.0, epsilon=1e-6):
    x_flat = x.detach().cpu().numpy().flatten()
    saturated_min = x_flat <= clip_min + epsilon
    saturated_max = x_flat >= clip_max - epsilon
    saturated = saturated_min | saturated_max
    return search_space & ~saturated

shape = (1, 1, 28, 28)
search_space = initialize_search_space(shape)
print('Initial shape:', search_space.shape)
print('Initial available pixels:', search_space.sum())
x_toy = torch.tensor([[[[0.0, 0.3], [0.95, 1.0]]]])
toy_mask = np.ones(4, dtype=bool)
updated_mask = remove_saturated_pixels(toy_mask, x_toy)
print('Toy pixel values:', x_toy.flatten().numpy())
print('Remaining mask:', updated_mask)

`shape[1:]` ignores the batch dimension and multiplies channels, height, and width: for MNIST, `1 × 28 × 28 = 784`. A small epsilon (say **epsilon**, a tolerance) treats values extremely close to 0 or 1 as saturated. The next HTB section uses these scores and masks inside the single-pixel attack loop.

## Single-pixel attack utilities

This section connects the math to an actual image. Saliency returns a flat pixel index, so we flatten, modify one value, clamp it to the valid image range, and reshape back. We also need a target-success test and a probability-based progress metric.

In [ ]:
def apply_single_pixel_perturbation(x, pixel_idx, theta, increase, clip_min=0.0, clip_max=1.0):
    original_shape = x.shape
    x_flat = x.view(-1).clone()
    perturbation = theta if increase else -theta
    x_flat[pixel_idx] = torch.clamp(
        x_flat[pixel_idx] + perturbation, clip_min, clip_max
    )
    return x_flat.view(original_shape)

def check_target_reached(x, target_class, model):
    with torch.no_grad():
        prediction = int(model(x).argmax(dim=1).item())
    return prediction == target_class

def compute_confidence(x, target_class, model):
    with torch.no_grad():
        probs = F.softmax(model(x), dim=1)
        return float(probs[0, target_class].item())

`theta` (say **theta**) is the signed step size. `torch.clamp` enforces `[clip_min, clip_max]`. `check_target_reached` uses `argmax` (argument max) on logits, while `compute_confidence` applies softmax so the target's progress is expressed as a probability.

In [ ]:
# Select one correctly classified MNIST image and choose a different target.
for x_batch, y_batch in test_loader:
    x_batch, y_batch = x_batch.to(device), y_batch.to(device)
    with torch.no_grad():
        preds = model(x_batch).argmax(dim=1)
    for i in range(x_batch.size(0)):
        if preds[i].item() == y_batch[i].item():
            x = x_batch[i:i+1]
            original_class = int(y_batch[i].item())
            target_class = (original_class + 5) % 10
            break
    break

x_adv = x.clone().detach()
theta = 0.25
clip_min, clip_max = 0.0, 1.0
search_space = initialize_search_space(x.shape)
print(f'Sample: digit {original_class}, target {target_class}')
print(f'Initial target confidence: {compute_confidence(x_adv, target_class, model):.4f}')

## One complete iteration

The following cells intentionally expose the data flow instead of hiding it in one large function: Jacobian → alpha/beta → masked saliency → winning pixel → perturbation → updated mask → success check. This is the final boundary of this subsection; the next subsection packages the same steps into a reusable attack loop.

In [ ]:
jacobian = compute_jacobian_matrix(x_adv, model, num_classes=10, wrt='logits')
alpha = apply_search_mask(extract_target_gradient(jacobian, target_class), search_space)
beta = apply_search_mask(extract_other_gradients(jacobian, target_class), search_space)
inc_scores = score_increase_saliency(alpha, beta)
dec_scores = score_decrease_saliency(alpha, beta)
pixel_idx, saliency, increase = select_best_direction(inc_scores, dec_scores)

print('Jacobian shape:', jacobian.shape)
print('Valid increases:', int((inc_scores > 0).sum()))
print('Valid decreases:', int((dec_scores > 0).sum()))
print(f'Selected pixel {pixel_idx}, saliency {saliency:.6f}, {"increase" if increase else "decrease"}')

if saliency > 0:
    pixel_before = x_adv.view(-1)[pixel_idx].item()
    x_adv = apply_single_pixel_perturbation(
        x_adv, pixel_idx, theta, increase, clip_min, clip_max
    )
    pixel_after = x_adv.view(-1)[pixel_idx].item()
    search_space = remove_saturated_pixels(
        search_space, x_adv, clip_min, clip_max
    )
    print(f'Pixel value: {pixel_before:.4f} -> {pixel_after:.4f}')
    print('Target reached:', check_target_reached(x_adv, target_class, model))
    print('Target confidence:', f'{compute_confidence(x_adv, target_class, model):.4f}')
else:
    print('No valid saliency score; stop the attack.')